In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
B = 16
T = 12
d_k = 4
w = 4

In [ ]:

Q = torch.randint(0,2,size = (1,T,d_k))
K = torch.randint(0,2,size = (1,T,d_k))
V = torch.randint(0,2,size = (1,T,d_k))

Q_chunks = [Q[:,j:j+w,:].float() for j in range(0,T,w)] #Q_chunk[0] : (1,w,d_k)
K_chunks = [K[:,j:j+w,:].float() for j in range(0,T,w)] 
V_chunks = [V[:,j:j+w,:].float() for j in range(0,T,w)]

mask_curr = torch.triu(torch.ones(w,w), diagonal=1).bool()
mask_prev= torch.triu(torch.ones(w,w), diagonal=1).bool()

chunk_curr = torch.stack([(F.softmax((Q_chunks[i]@K_chunks[i].transpose(-2,-1)/d_k**0.5).masked_fill(mask_curr, float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i])[0] for i in range(len(Q_chunks))])
chunk_prev = torch.stack([(F.softmax((Q_chunks[i]@K_chunks[i-1].transpose(-2,-1)/d_k**0.5).masked_fill(~mask_prev,float('-inf')),dim=-1,dtype = torch.float)@V_chunks[i])[0] for i in range(1,len(Q_chunks))])

chunk0 = chunk_curr[0]

chunk_i = [chunk_curr[i+1]+chunk_prev[i] for i in range(0,(chunk_prev.shape)[0])]

res = torch.cat((chunk0,*chunk_i),dim=0).nan_to_num(0).reshape(1,12,4)


torch.Size([1, 12, 4])